# Construcción de la matriz de diseño

**EC2053C · Análisis de Datos II · Ayudantía 02 · Semana 2**
Universidad Católica de la Santísima Concepción · FACEA · Ingeniería en Información y Control de Gestión

---

Este cuaderno lleva un extracto transaccional —muchas filas por cliente— a una **matriz de diseño**: una fila
por unidad de decisión, con todas las variables congeladas al instante en que la decisión se toma, y una
etiqueta explícita.

El caso de demostración es una **distribuidora B2B** que cada mes decide a qué clientes anticipar gestión de
cobranza. Reemplace la sección 1 por el extracto de su propia organización y el resto del cuaderno funciona
igual: la estructura no cambia, cambian los datos.

### Las cuatro decisiones de diseño

| Decisión | Valor en este caso | Por qué |
|---|---|---|
| **Unidad de análisis** | Cliente | La gestión de cobranza se asigna por cliente, no por factura. |
| **Grano temporal** | Mensual | La cartera se revisa una vez al mes, en el cierre. |
| **Ventana de observación** | 180 días antes de `t₀` | Cubre dos ciclos completos de facturación y compra. |
| **Ventana de predicción** | 30 días después de `t₀` | Horizonte en que la gestión todavía alcanza a evitar la mora. |

**Regla que ordena todo el cuaderno:** ninguna variable puede usar información que no existiera, con ese valor,
el día `t₀`. La sección 11 audita esa regla y muestra qué pasa cuando se rompe.

## 0. Configuración

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 60)

SEED = 2053
rng = np.random.default_rng(SEED)

# --- parámetros de diseño (los únicos que se tocan al cambiar de caso) ---
T0            = pd.Timestamp("2026-06-30")   # fecha de corte: el instante en que se decide
VENT_OBS      = pd.Timedelta(days=180)       # ventana de observación hacia atrás
VENT_PRED     = pd.Timedelta(days=30)        # ventana de predicción hacia adelante
TOLERANCIA    = 5                            # días de atraso que el negocio considera aceptables

INICIO_OBS    = T0 - VENT_OBS
FIN_PRED      = T0 + VENT_PRED

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
for c in ["data/raw", "data/interim", "data/processed", "reports"]:
    (RAIZ / c).mkdir(parents=True, exist_ok=True)

print(f"Ventana de observación : {INICIO_OBS:%Y-%m-%d}  →  {T0:%Y-%m-%d}")
print(f"Corte t0               : {T0:%Y-%m-%d}")
print(f"Ventana de predicción  : {T0:%Y-%m-%d}  →  {FIN_PRED:%Y-%m-%d}")
print(f"Raíz del proyecto      : {RAIZ}")

## 1. Datos de demostración

> **Reemplace esta sección completa por la lectura del extracto de su organización.** El resto del cuaderno
> sólo necesita dos tablas: `clientes` (una fila por cliente) y `facturas` (una fila por documento, con fecha de
> emisión, fecha de vencimiento, monto y fecha de pago cuando existe).

Los datos se generan con la semilla del curso, de modo que todos obtienen exactamente la misma matriz y pueden
comparar resultados entre equipos.

In [ ]:
COMUNAS = {
    "Concepción": "Centro", "Talcahuano": "Centro", "San Pedro de la Paz": "Centro",
    "Hualpén": "Centro", "Chiguayante": "Centro",
    "Los Ángeles": "Interior", "Chillán": "Interior", "Cañete": "Interior",
    "Arauco": "Costa", "Coronel": "Costa", "Lota": "Costa", "Lebu": "Costa",
}
SEGMENTOS = ["Retail", "Mayorista", "Institucional"]
CANALES   = ["Terreno", "Televenta", "Ecommerce"]

N_CLIENTES = 600
FECHA_EXTRACCION = pd.Timestamp("2026-08-31")   # cuándo se bajó el extracto del sistema

# ---------- tabla de clientes ----------
comunas = list(COMUNAS)
clientes = pd.DataFrame({
    "cliente_id": [f"C{i:04d}" for i in range(1, N_CLIENTES + 1)],
    "fecha_alta": pd.Timestamp("2019-01-01") + pd.to_timedelta(
        rng.integers(0, 2600, size=N_CLIENTES), unit="D"),
    "segmento":   rng.choice(SEGMENTOS, size=N_CLIENTES, p=[0.55, 0.30, 0.15]),
    "comuna":     rng.choice(comunas, size=N_CLIENTES),
    "canal":      rng.choice(CANALES, size=N_CLIENTES, p=[0.45, 0.35, 0.20]),
})

# propensión latente al atraso: NO forma parte de la matriz, sólo genera los datos
propension = rng.beta(2, 4, size=N_CLIENTES)
propension += np.where(clientes["segmento"].to_numpy() == "Institucional", -0.08, 0.0)
propension += np.where(clientes["canal"].to_numpy() == "Terreno", 0.05, 0.0)
propension = np.clip(propension, 0.01, 0.95)
prop = dict(zip(clientes["cliente_id"], propension))

escala = dict(zip(clientes["cliente_id"],
                  rng.lognormal(mean=14.2, sigma=0.55, size=N_CLIENTES)))
actividad = dict(zip(clientes["cliente_id"], rng.uniform(0.45, 0.95, size=N_CLIENTES)))

# ---------- tabla de facturas ----------
meses = pd.date_range("2025-07-01", "2026-08-01", freq="MS")
filas = []
k = 0
for cid in clientes["cliente_id"]:
    for m in meses:
        if rng.random() > actividad[cid]:
            continue
        for _ in range(int(rng.integers(1, 4))):
            k += 1
            emision = m + pd.Timedelta(days=int(rng.integers(0, 28)))
            vence   = emision + pd.Timedelta(days=30)
            monto   = float(escala[cid] * rng.lognormal(0, 0.35))
            # días de atraso: cola larga gobernada por la propensión del cliente
            atraso = int(rng.negative_binomial(2, 1 - min(prop[cid] * 0.82, 0.9)))
            impaga = rng.random() < prop[cid] * 0.14
            pago = pd.NaT if impaga else vence + pd.Timedelta(days=atraso)
            if pd.notna(pago) and pago > FECHA_EXTRACCION:
                pago = pd.NaT                      # aún no pagada al momento del extracto
            filas.append((f"F{k:06d}", cid, emision, vence, round(monto, 0), pago))

facturas = pd.DataFrame(filas, columns=[
    "factura_id", "cliente_id", "fecha_emision", "fecha_vencimiento", "monto", "fecha_pago"])

clientes.to_csv(RAIZ / "data/raw/clientes.csv", index=False)
facturas.to_csv(RAIZ / "data/raw/facturas.csv", index=False)

print(f"clientes : {len(clientes):>6,} filas")
print(f"facturas : {len(facturas):>6,} filas   ({facturas['fecha_emision'].min():%Y-%m-%d} a "
      f"{facturas['fecha_emision'].max():%Y-%m-%d})")
facturas.head()

## 2. Carga y tipificación

Primera regla del extracto: **las fechas son fechas, no texto**. Un `object` donde debería haber un `datetime64`
es la causa más frecuente de comparaciones que fallan en silencio y de ventanas mal calculadas.

In [ ]:
clientes = pd.read_csv(RAIZ / "data/raw/clientes.csv", parse_dates=["fecha_alta"])
facturas = pd.read_csv(RAIZ / "data/raw/facturas.csv",
                       parse_dates=["fecha_emision", "fecha_vencimiento", "fecha_pago"])

for nombre, df in [("clientes", clientes), ("facturas", facturas)]:
    print(f"--- {nombre}  {df.shape[0]:,} × {df.shape[1]}")
    print(df.dtypes.to_string(), "\n")

print("Facturas sin fecha de pago al momento del extracto:",
      f"{facturas['fecha_pago'].isna().sum():,} "
      f"({facturas['fecha_pago'].isna().mean():.1%})")

## 3. Congelar `t₀`

Aquí se parte el mundo en dos y no se vuelve a mezclar:

- **`obs`** — todo lo que el sistema ya había registrado al cierre del 30 de junio de 2026. Es lo único que puede
  alimentar variables.
- **`fut`** — lo que ocurrió después. Sirve **únicamente** para construir la etiqueta.

Todo lo que se calcule a partir de la sección 4 usa `obs`. El `assert` no es decorativo: es la barrera que
impide que un `groupby` distraído lea el futuro.

In [ ]:
# Una factura es "conocida en t0" si ya fue emitida en t0.
obs = facturas.loc[facturas["fecha_emision"] <= T0].copy()

# El pago sólo se conoce si ocurrió antes de t0. Si se pagó después, en t0 estaba impaga.
obs["pago_conocido_en_t0"] = obs["fecha_pago"].where(obs["fecha_pago"] <= T0)

fut = facturas.loc[facturas["fecha_emision"] <= T0].copy()   # la etiqueta usa facturas ya emitidas...
fut = fut.loc[(fut["fecha_vencimiento"] > T0) & (fut["fecha_vencimiento"] <= FIN_PRED)]  # ...que vencen después

print(f"Facturas emitidas hasta t0        : {len(obs):,}")
print(f"  de ellas, pagadas antes de t0   : {obs['pago_conocido_en_t0'].notna().sum():,}")
print(f"Facturas que vencen en la ventana : {len(fut):,}  "
      f"({fut['cliente_id'].nunique():,} clientes)")

assert obs["fecha_emision"].max() <= T0, "Hay facturas posteriores a t0 en el conjunto de observación."
assert obs["pago_conocido_en_t0"].max() <= T0, "Hay pagos posteriores a t0 marcados como conocidos."
print("\nOK: no hay información posterior a t0 en el conjunto de observación.")

## 4. Definir la población

La matriz **no** contiene a todos los clientes: contiene a aquellos sobre los que hay una decisión que tomar.
Un cliente sin facturas por vencer en la ventana no entra, porque no hay nada que decidir sobre él.

Esta elección define la población de estudio y hay que declararla en el informe. Cambiarla cambia la prevalencia
de la etiqueta y, con ella, todas las métricas.

In [ ]:
poblacion = (fut[["cliente_id"]]
             .drop_duplicates()
             .assign(fecha_corte=T0)
             .sort_values("cliente_id")
             .reset_index(drop=True))

print(f"Clientes en la base            : {len(clientes):,}")
print(f"Clientes con decisión en t0    : {len(poblacion):,}  "
      f"({len(poblacion)/len(clientes):.1%} de la base)")
poblacion.head()

## 5. Bloque A — atributos del cliente en `t₀`

Atributos estables o calculables a la fecha de corte. `antiguedad_meses` se calcula **contra `t₀`**, no contra
hoy: si se calculara contra la fecha de ejecución del cuaderno, la variable cambiaría de valor cada vez que se
corre, y el modelo dejaría de ser reproducible.

In [ ]:
bloque_a = poblacion.merge(clientes, on="cliente_id", how="left")

bloque_a["antiguedad_meses"] = ((T0 - bloque_a["fecha_alta"]).dt.days / 30.44).round(1)
bloque_a["zona"] = bloque_a["comuna"].map(COMUNAS)

bloque_a = bloque_a[["cliente_id", "fecha_corte", "antiguedad_meses", "segmento", "canal", "zona"]]
print(bloque_a.dtypes.to_string())
bloque_a.head()

## 6. Bloque B — recencia, frecuencia y monto

El esqueleto de casi cualquier matriz comercial. Todo se calcula sobre la ventana de observación
`[t₀ − 180 días, t₀]`, la misma para todos los clientes: si cada fila usara una ventana distinta, las variables
no serían comparables entre sí.

In [ ]:
obs_180 = obs.loc[obs["fecha_emision"].between(INICIO_OBS, T0)]

bloque_b = (obs_180.groupby("cliente_id")
            .agg(n_facturas_180d   = ("factura_id", "size"),
                 monto_total_180d  = ("monto", "sum"),
                 ticket_medio_180d = ("monto", "mean"),
                 sd_monto_180d     = ("monto", "std"),
                 ultima_emision    = ("fecha_emision", "max"))
            .reset_index())

bloque_b["recencia_dias"] = (T0 - bloque_b["ultima_emision"]).dt.days
bloque_b["sd_monto_180d"] = bloque_b["sd_monto_180d"].fillna(0.0)
bloque_b = bloque_b.drop(columns="ultima_emision")

print(bloque_b[["n_facturas_180d", "monto_total_180d", "ticket_medio_180d",
                "recencia_dias"]].describe().round(1).to_string())
bloque_b.head()

## 7. Bloque C — comportamiento de pago observable en `t₀`

Aquí está el punto fino de toda la sesión. El atraso de una factura **no** es
`fecha_pago − fecha_vencimiento`: eso sólo se sabe una vez que la factura se paga, y muchas veces el pago ocurre
después de `t₀`.

Lo que se observa el 30 de junio es:

- si la factura ya se pagó antes de `t₀` → el atraso real, `fecha_pago − vencimiento`;
- si todavía no se ha pagado en `t₀` → el atraso **acumulado hasta hoy**, `t₀ − vencimiento`, que es información
  legítima y además muy predictiva.

Usar el atraso final de una factura pagada en agosto para predecir algo en julio es fuga de información, y es
exactamente el error que la sección 11 hace visible.

In [ ]:
venc = obs.loc[obs["fecha_vencimiento"] <= T0].copy()          # ya exigibles en t0
venc = venc.loc[venc["fecha_vencimiento"] >= INICIO_OBS]        # dentro de la ventana

pagada_antes = venc["pago_conocido_en_t0"].notna()
venc["atraso_obs"] = np.where(
    pagada_antes,
    (venc["pago_conocido_en_t0"] - venc["fecha_vencimiento"]).dt.days,
    (T0 - venc["fecha_vencimiento"]).dt.days,
).clip(min=0)
venc["impaga_en_t0"] = (~pagada_antes).astype(int)
venc["atrasada"] = (venc["atraso_obs"] > TOLERANCIA).astype(int)

bloque_c = (venc.groupby("cliente_id")
            .agg(n_venc_180d          = ("factura_id", "size"),
                 atraso_medio_180d    = ("atraso_obs", "mean"),
                 atraso_max_180d      = ("atraso_obs", "max"),
                 pct_atrasadas_180d   = ("atrasada", "mean"),
                 n_impagas_en_t0      = ("impaga_en_t0", "sum"))
            .reset_index())

vencido = (venc.loc[venc["impaga_en_t0"] == 1]
           .groupby("cliente_id")["monto"].sum()
           .rename("monto_vencido_en_t0").reset_index())

bloque_c = bloque_c.merge(vencido, on="cliente_id", how="left")
bloque_c["monto_vencido_en_t0"] = bloque_c["monto_vencido_en_t0"].fillna(0.0)
bloque_c[["atraso_medio_180d", "pct_atrasadas_180d"]] = \
    bloque_c[["atraso_medio_180d", "pct_atrasadas_180d"]].round(3)

print(bloque_c.describe().round(2).to_string())
bloque_c.head()

## 8. Bloque D — tendencia y contexto relativo

Dos familias que casi nunca vienen en el sistema y que suelen aportar más que el nivel absoluto:

- **Tendencia**: la razón entre lo facturado en los últimos 60 días y el promedio bimestral de la ventana.
  Un valor bajo 1 indica que el cliente se está enfriando.
- **Contexto relativo**: el monto del cliente dividido por la mediana de su segmento. Un monto de 5 millones
  significa cosas distintas en Retail y en Mayorista.

La mediana del segmento se calcula **sólo con datos anteriores a `t₀`**. Calcularla con toda la tabla, futuro
incluido, es una fuga sutil y muy común.

In [ ]:
obs_60 = obs.loc[obs["fecha_emision"].between(T0 - pd.Timedelta(days=60), T0)]
monto_60 = obs_60.groupby("cliente_id")["monto"].sum().rename("monto_60d")

bloque_d = bloque_b[["cliente_id", "monto_total_180d"]].merge(
    monto_60, on="cliente_id", how="left")
bloque_d["monto_60d"] = bloque_d["monto_60d"].fillna(0.0)

promedio_bimestral = bloque_d["monto_total_180d"] / 3
bloque_d["razon_60d_vs_promedio"] = np.where(
    promedio_bimestral > 0, bloque_d["monto_60d"] / promedio_bimestral, np.nan).round(3)

# mediana por segmento, calculada sólo con la ventana de observación
seg = bloque_a[["cliente_id", "segmento"]]
tmp = bloque_d.merge(seg, on="cliente_id", how="left")
mediana_seg = tmp.groupby("segmento")["monto_total_180d"].median().rename("mediana_segmento")
tmp = tmp.merge(mediana_seg, on="segmento", how="left")
bloque_d["monto_vs_mediana_segmento"] = (
    tmp["monto_total_180d"] / tmp["mediana_segmento"]).round(3)

bloque_d = bloque_d[["cliente_id", "razon_60d_vs_promedio", "monto_vs_mediana_segmento"]]
print(mediana_seg.round(0).to_string(), "\n")
bloque_d.head()

## 9. Bloque E — calendario y exposición conocida

Variables que se conocen con anticipación y que por definición no pueden tener fuga: el calendario no depende de
lo que hagan los clientes.

`n_facturas_por_vencer` merece una nota: cuenta documentos **ya emitidos antes de `t₀`** cuyo vencimiento cae en
la ventana de predicción. Aunque mira hacia adelante en el tiempo, era perfectamente conocida el 30 de junio, y
por eso es admisible. La distinción no es *pasado contra futuro*, sino *conocido contra desconocido en `t₀`*.

In [ ]:
exposicion = (fut.groupby("cliente_id")
              .agg(n_facturas_por_vencer = ("factura_id", "size"),
                   monto_por_vencer      = ("monto", "sum"))
              .reset_index())

bloque_e = poblacion[["cliente_id"]].merge(exposicion, on="cliente_id", how="left")

bloque_e["mes_corte"]      = T0.month
bloque_e["trimestre_corte"] = T0.quarter
bloque_e["dias_habiles_ventana"] = int(np.busday_count(
    (T0 + pd.Timedelta(days=1)).date(), (FIN_PRED + pd.Timedelta(days=1)).date()))

# feriados legales de Chile en el segundo semestre de 2026
FERIADOS = pd.to_datetime(["2026-08-15", "2026-09-18", "2026-09-19", "2026-10-12",
                           "2026-10-31", "2026-11-01", "2026-12-08", "2026-12-25"])
bloque_e["feriados_en_ventana"] = int(
    ((FERIADOS > T0) & (FERIADOS <= FIN_PRED)).sum())

print(bloque_e[["n_facturas_por_vencer", "monto_por_vencer"]].describe().round(1).to_string())
print("\nDías hábiles en la ventana de predicción:", bloque_e["dias_habiles_ventana"].iloc[0])
print("Feriados legales en la ventana          :", bloque_e["feriados_en_ventana"].iloc[0])
bloque_e.head()

## 10. La etiqueta

**`mora_30d = 1`** si el cliente tiene al menos una factura que vence en `(t₀, t₀ + 30 días]` y que quedó impaga
o se pagó con más de 5 días de atraso.

Tres decisiones que hay que poder defender en el informe, porque cambian el problema:

1. **El umbral de 5 días** viene del negocio, no de los datos: es la holgura administrativa que la empresa ya
   considera normal.
2. **«Al menos una»** convierte el problema en binario por cliente. Alternativa razonable: la proporción del
   monto en mora, que sería un problema de regresión.
3. **Impaga cuenta como mora.** Una factura sin pago registrado al momento del extracto es, para efectos de la
   gestión, exactamente el caso que se quiere anticipar.

In [ ]:
fut = fut.copy()
fut["en_mora"] = (
    fut["fecha_pago"].isna() |
    ((fut["fecha_pago"] - fut["fecha_vencimiento"]).dt.days > TOLERANCIA)
).astype(int)

etiqueta = (fut.groupby("cliente_id")["en_mora"].max()
            .rename("mora_30d").reset_index())

print(etiqueta["mora_30d"].value_counts().rename({0: "sin mora", 1: "en mora"}).to_string())
print(f"\nPrevalencia: {etiqueta['mora_30d'].mean():.1%}")
print("\nUna prevalencia de este orden ya obliga a mirar precisión-exhaustividad y no sólo exactitud.")

## 11. Ensamblaje

Se unen los bloques por la izquierda, sobre la población definida en la sección 4. Las ausencias que aparezcan
en el camino no se rellenan a ciegas: se documentan y se tratan según lo que significan.

In [ ]:
matriz = (poblacion
          .merge(bloque_a, on=["cliente_id", "fecha_corte"], how="left")
          .merge(bloque_b, on="cliente_id", how="left")
          .merge(bloque_c, on="cliente_id", how="left")
          .merge(bloque_d, on="cliente_id", how="left")
          .merge(bloque_e, on="cliente_id", how="left")
          .merge(etiqueta, on="cliente_id", how="left"))

# ausencias con significado conocido: "no ocurrió" es 0, no es "se desconoce"
CERO_ES_CORRECTO = ["n_facturas_180d", "monto_total_180d", "sd_monto_180d",
                    "n_venc_180d", "n_impagas_en_t0", "monto_vencido_en_t0",
                    "n_facturas_por_vencer", "monto_por_vencer"]
matriz[CERO_ES_CORRECTO] = matriz[CERO_ES_CORRECTO].fillna(0)

# el resto se deja como NaN: el imputador del pipeline lo resuelve dentro del pliegue
faltantes = matriz.isna().sum()
faltantes = faltantes[faltantes > 0]

print(f"Matriz: {matriz.shape[0]:,} filas × {matriz.shape[1]} columnas\n")
print("Columnas con valores faltantes (se imputan dentro del pipeline, nunca aquí):")
print(faltantes.to_string() if len(faltantes) else "  ninguna")
matriz.head()

## 12. Auditoría de fuga de información

Dos comprobaciones. La primera es mecánica y debería estar en todo cuaderno de este tipo. La segunda es la
demostración de por qué importa.

### 12.1 Auditoría estructural

In [ ]:
PROHIBIDAS = ["fecha_pago", "recuperado", "post", "final", "cierre", "resultado"]

sospechosas = [c for c in matriz.columns
               if any(p in c.lower() for p in PROHIBIDAS) and c != "mora_30d"]

print("Columnas con nombre sospechoso:", sospechosas or "ninguna")
print("Correlación de cada variable numérica con la etiqueta:\n")

numericas = matriz.select_dtypes("number").drop(columns=["mora_30d"])
constantes = [c for c in numericas.columns if numericas[c].nunique() <= 1]
corr = (numericas.drop(columns=constantes)
        .corrwith(matriz["mora_30d"])
        .sort_values(key=abs, ascending=False))
print(corr.round(3).to_string())

print("\nColumnas constantes en esta matriz (sin correlación definida):", constantes)
print("No son un error: con un solo t0, el calendario no varía. Empiezan a informar")
print("cuando la matriz apila varios cortes mensuales, que es lo que haremos en la semana 13.")

print("\nRegla práctica: una correlación individual sobre 0,80 en un problema de negocio")
print("no es una gran variable, es una hipótesis de fuga que hay que descartar.")

### 12.2 Qué pasa cuando se rompe la regla

Construimos a propósito una variable contaminada, `monto_recuperado_post`, que sólo se conoce **después** de la
ventana de predicción, y comparamos el desempeño con y sin ella bajo exactamente el mismo protocolo.

El punto no es que el AUC suba: es *cuánto* sube, y lo convincente que resulta el resultado antes de que alguien
pregunte de dónde salió esa columna.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold

# variable contaminada: monto efectivamente cobrado dentro de la ventana (información del futuro)
recuperado = (fut.loc[fut["en_mora"] == 0]
              .groupby("cliente_id")["monto"].sum()
              .rename("monto_recuperado_post").reset_index())
sucia = matriz.merge(recuperado, on="cliente_id", how="left")
sucia["monto_recuperado_post"] = sucia["monto_recuperado_post"].fillna(0.0)

CAT = ["segmento", "canal", "zona"]

def evaluar(df, etiqueta_col="mora_30d"):
    X = df.drop(columns=[etiqueta_col, "cliente_id", "fecha_corte"])
    X = X.drop(columns=[c for c in X.columns if X[c].nunique() <= 1])   # constantes: sin información
    y = df[etiqueta_col]
    num = [c for c in X.columns if c not in CAT]
    pre = ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                          ("esc", StandardScaler())]), num),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("oh",  OneHotEncoder(handle_unknown="ignore"))]), CAT),
    ])
    modelo = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=2000))])
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    return cross_val_score(modelo, X, y, cv=cv, scoring="roc_auc")

auc_limpia = evaluar(matriz)
auc_sucia  = evaluar(sucia)

print(f"Matriz limpia (sólo información disponible en t0)")
print(f"   AUC = {auc_limpia.mean():.3f}  ±{auc_limpia.std():.3f}")
print(f"\nMatriz contaminada (+ monto_recuperado_post)")
print(f"   AUC = {auc_sucia.mean():.3f}  ±{auc_sucia.std():.3f}")
print(f"\nGanancia aparente: +{auc_sucia.mean() - auc_limpia.mean():.3f} de AUC.")
print("Ganancia real en producción: cero. Esa columna no existe el día que hay que decidir.")

## 13. Guardar la matriz y el diccionario

El archivo `matriz_diseno.parquet` es el insumo de las semanas 3 a 18. Parquet y no CSV por tres razones: conserva
los tipos (las fechas siguen siendo fechas), ocupa mucho menos, y se lee más rápido.

El diccionario de variables se genera desde la propia matriz, con una columna de disponibilidad en `t₀` que hay
que revisar a mano. Ese es el campo que se corrige primero en la Entrega 02.

In [ ]:
destino = RAIZ / "data/processed/matriz_diseno.parquet"
matriz.to_parquet(destino, index=False)

DEFINICIONES = {
    "cliente_id":                "Identificador del cliente. Clave de la matriz.",
    "fecha_corte":               "Fecha t0 en que se toma la decisión de gestión.",
    "antiguedad_meses":          "Meses transcurridos entre el alta del cliente y t0.",
    "segmento":                  "Segmento comercial: Retail, Mayorista o Institucional.",
    "canal":                     "Canal de atención principal del cliente.",
    "zona":                      "Agrupación geográfica de la comuna.",
    "n_facturas_180d":           "Facturas emitidas en los 180 días previos a t0.",
    "monto_total_180d":          "Monto facturado en los 180 días previos a t0.",
    "ticket_medio_180d":         "Monto promedio por factura en la ventana de observación.",
    "sd_monto_180d":             "Desviación estándar del monto facturado en la ventana.",
    "recencia_dias":             "Días entre la última emisión y t0.",
    "n_venc_180d":               "Facturas exigibles en la ventana de observación.",
    "atraso_medio_180d":         "Atraso promedio observable en t0, en días.",
    "atraso_max_180d":           "Atraso máximo observable en t0, en días.",
    "pct_atrasadas_180d":        "Proporción de facturas con atraso mayor a la tolerancia.",
    "n_impagas_en_t0":           "Facturas vencidas y sin pago registrado en t0.",
    "monto_vencido_en_t0":       "Monto vencido e impago acumulado en t0.",
    "razon_60d_vs_promedio":     "Facturación de los últimos 60 días sobre el promedio bimestral.",
    "monto_vs_mediana_segmento": "Monto del cliente sobre la mediana de su segmento en la ventana.",
    "n_facturas_por_vencer":     "Facturas ya emitidas en t0 que vencen en la ventana de predicción.",
    "monto_por_vencer":          "Monto de esas facturas. Exposición conocida en t0.",
    "mes_corte":                 "Mes calendario de t0.",
    "trimestre_corte":           "Trimestre calendario de t0.",
    "dias_habiles_ventana":      "Días hábiles en la ventana de predicción.",
    "feriados_en_ventana":       "Feriados legales de Chile en la ventana de predicción.",
    "mora_30d":                  "ETIQUETA. 1 si alguna factura que vence en la ventana queda impaga o se paga "
                                 "con más de 5 días de atraso.",
}

diccionario = pd.DataFrame({
    "variable":     matriz.columns,
    "tipo":         [str(t) for t in matriz.dtypes],
    "definicion":   [DEFINICIONES.get(c, "") for c in matriz.columns],
    "faltantes":    matriz.isna().sum().to_numpy(),
    "disponible_t0": ["etiqueta" if c == "mora_30d" else "sí" for c in matriz.columns],
})
diccionario.to_csv(RAIZ / "reports/diccionario_variables.csv", index=False, encoding="utf-8")

print("Guardado:")
print("  ", destino, f"({destino.stat().st_size/1024:.1f} KB)")
print("  ", RAIZ / "reports/diccionario_variables.csv")
print(f"\nMatriz final: {matriz.shape[0]:,} filas × {matriz.shape[1]} columnas, "
      f"prevalencia {matriz['mora_30d'].mean():.1%}")
diccionario

## 14. Verificación de la lectura

Se vuelve a leer el archivo guardado y se comprueba que los tipos sobrevivieron. Un parquet que se lee distinto
de como se escribió rompe la reproducibilidad en la semana siguiente, no en esta.

In [ ]:
leida = pd.read_parquet(RAIZ / "data/processed/matriz_diseno.parquet")

print("Dimensiones idénticas :", leida.shape == matriz.shape)
print("Tipos idénticos       :", (leida.dtypes == matriz.dtypes).all())
print("Etiqueta idéntica     :", bool((leida["mora_30d"] == matriz["mora_30d"]).all()))
print(f"\n{leida.shape[0]:,} filas × {leida.shape[1]} columnas")
leida.describe(include="all").T.head(30)

---

## Qué se entrega

| Elemento | Dónde | Plazo |
|---|---|---|
| `02_matriz_diseno.ipynb` ejecutado, con las salidas visibles | Repositorio del equipo, con *commit* fechado | domingo 16 de agosto, 23:59 |
| `data/processed/matriz_diseno.parquet` | Repositorio del equipo | domingo 16 de agosto, 23:59 |
| Diccionario de variables (Entrega 02) | EV@ | domingo 16 de agosto, 23:59 |

El diccionario que genera la sección 13 es un punto de partida: la columna **`disponible_t0`** hay que revisarla
variable por variable y justificarla en una línea. Ese criterio pesa 40% de la Entrega 02, porque una variable
mal justificada se corrige y una variable con fuga obliga a rehacer todo lo que venga después.

### Para la semana 3

Traiga identificada **una variable de su propio diccionario que sospeche que tiene fuga**, y la razón de la
sospecha. En el laboratorio vamos a diagnosticarla con el mismo procedimiento de la sección 12.